# Experiment 2b — Stage-2 fine-tune: Mistral-7B-v0.1 (Kaggle) · cascade-pid-13g.3

Kaggle port of `notebooks/03_stage2_finetune.ipynb`. Fine-tunes
**Mistral-7B-v0.1** (same base weights as the released DataSentinel-7B) into an
in-distribution benign/injection `p_safe` classifier on the length-augmented
`data/train_proposal/train_stage2.jsonl` (13g.2), via `scripts/train_stage1.py`
+ `configs/training_7b.yaml` (batch 1 × grad-accum 32).

**Setup (Kaggle UI):** same as the stage-1 notebook — add the `cascade_kaggle_repo`
dataset, **GPU accelerator**, **Internet ON**, `HF_TOKEN` secret, and accept the
Mistral-7B-v0.1 license on huggingface.co.

**VRAM:** 7B QLoRA at 4-bit is ~8–11 GB with batch 1 + grad-checkpointing @ seq 2048,
so a **Kaggle T4 (16 GB) fits it** (tight but OK). Outputs →
`/kaggle/working/results/stage2/mistral-7b-v0.1/`.

**Payload hygiene:** counts / lengths / p_safe / paths only.

In [ ]:
# ── Environment detection + installs (Kaggle) ───────────────────────────
# Requires: Notebook settings -> Accelerator = GPU (T4 x2 or P100),
#           Notebook settings -> Internet = ON (for pip + HF downloads).
import os, sys
IN_KAGGLE = os.path.exists("/kaggle/input")
if IN_KAGGLE:
    !pip install -q -U bitsandbytes accelerate peft transformers trl datasets
    print("Kaggle detected — training deps installed.")
    print("  (If pip failed: Notebook settings -> Internet must be ON.)")
else:
    print(f"Local run ({sys.platform}) — using the project environment as-is.")


In [ ]:
# ── Config (Kaggle) ────────────────────────────────────────────────
RUN_MODE = "smoke"   # "smoke" | "full"   <- set "full" once GPU + token are confirmed
SEED     = 3131
import os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")
if IN_KAGGLE:
    # Auto-find the uploaded repo dataset at ANY depth under /kaggle/input (Kaggle
    # sometimes mounts a dataset as /kaggle/input/datasets/<user>/<name>/...).
    _hits = list(Path("/kaggle/input").rglob("qwen2.5-1.5b.yaml"))
    assert _hits, ("Repo dataset not found under /kaggle/input. Attach it: "
                   "Notebook -> Add Input -> your uploaded repo dataset.")
    REPO_ROOT   = _hits[0].parents[2]        # .../<root>/configs/models/*.yaml -> <root> (READ-ONLY)
    RESULTS_BASE = Path("/kaggle/working")   # writable + downloadable
    # HF token from Kaggle Secrets (Add-ons -> Secrets, name it HF_TOKEN)
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = _tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
        print("HF_TOKEN loaded from Kaggle Secrets.")
    except Exception as e:
        print("WARNING: HF_TOKEN not loaded — gated models (Llama/Granite/Mistral) will 403.", e)
else:
    _here = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "_")).resolve()
    REPO_ROOT = next((p for p in _here.parents if (p / "BASELINE_SPEC.md").exists()), Path.cwd())
    RESULTS_BASE = REPO_ROOT

for _p in (str(REPO_ROOT), str(REPO_ROOT / "src")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

BASE = "mistral-7b-v0.1"
CONFIG_PATH  = REPO_ROOT / "configs" / "models" / f"{BASE}.yaml"
TRAINING_CFG = REPO_ROOT / "configs" / "training_7b.yaml"
TRAIN_FILE   = REPO_ROOT / "data" / "train_proposal" / "train_stage2.jsonl"
VAL_FILE     = REPO_ROOT / "data" / "train_proposal" / "val.jsonl"
MANIFEST     = REPO_ROOT / "data" / "train_proposal" / "stage2_aug_manifest.json"
OUTPUT_DIR   = RESULTS_BASE / "results" / "stage2" / BASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert CONFIG_PATH.exists(), f"missing {CONFIG_PATH}"
assert TRAINING_CFG.exists(), f"missing {TRAINING_CFG}"
assert TRAIN_FILE.exists(), f"missing {TRAIN_FILE} (train_stage2.jsonl must be in the dataset)"

print(f"IN_KAGGLE={IN_KAGGLE}  RUN_MODE={RUN_MODE}  SEED={SEED}")
print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"RESULTS_BASE= {RESULTS_BASE}")


## Device detection

QLoRA 4-bit needs a CUDA GPU. On Kaggle set Accelerator=GPU. On CPU the notebook
falls back to a 1-step dry-run (no real training) so it won't OOM the host RAM.

In [ ]:
import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
CUDA_AVAILABLE = (DEVICE == "cuda")
DRY_RUN = not (CUDA_AVAILABLE and RUN_MODE == "full")
print(f"Detected device: {DEVICE}   CUDA={CUDA_AVAILABLE}   DRY_RUN={DRY_RUN}")
if not CUDA_AVAILABLE:
    print("\n" + "=" * 70)
    print("No CUDA GPU visible. On Kaggle: Notebook settings -> Accelerator -> GPU,")
    print("then Run all. Training a 1-2B model on CPU will OOM the host (SIGKILL -9).")
    print("Proceeding in DRY-RUN (1 step) only to validate the pipeline.")
    print("=" * 70)


In [ ]:
import subprocess
def run_train(config_path, out_dir, train_file=None, training_cfg=None):
    """Shell out to scripts/train_stage1.py; stream logs live into the notebook."""
    cmd = [sys.executable, "-u", str(REPO_ROOT / "scripts" / "train_stage1.py"),
           "--config", str(config_path), "--output-dir", str(out_dir)]
    if train_file is not None:
        cmd += ["--train-file", str(train_file)]
    if training_cfg is not None:
        cmd += ["--training", str(training_cfg)]
    if DRY_RUN:
        cmd += ["--dry-run"]
    print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=str(REPO_ROOT),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    return proc.returncode


In [ ]:
import json, math
def sanity_logits(out_dir, n_val=5724, n_cal=5722):
    """val/cal logit dumps match split row counts + p_safe range. Payload-safe."""
    report = {}
    for stem, n_exp in (("val", n_val), ("cal", n_cal)):
        fp = Path(out_dir) / f"{stem}_logits.jsonl"
        if not fp.exists():
            report[stem] = {"exists": False}; continue
        n = 0; bad = 0; pmin = 1.0; pmax = 0.0
        with fp.open() as fh:
            for line in fh:
                if not line.strip(): continue
                r = json.loads(line); n += 1; p = r.get("p_safe")
                if p is None or not (0.0 <= p <= 1.0) or math.isnan(p) \
                   or "logp_benign" not in r or "logp_injection" not in r:
                    bad += 1
                else:
                    pmin = min(pmin, p); pmax = max(pmax, p)
        report[stem] = {"exists": True, "rows": n, "expected": n_exp,
                        "rows_ok": n == n_exp, "bad_rows": bad,
                        "p_safe_min": round(pmin, 4), "p_safe_max": round(pmax, 4)}
    return report


In [ ]:
from evaluation.metrics import detection_rate_at_fpr
from collections import defaultdict
_SAFE = {"safe", "benign", 0, "0"}
def _load_val_meta(val_path):
    labels, channels = [], []
    with open(val_path) as fh:
        for line in fh:
            if not line.strip(): continue
            r = json.loads(line)
            labels.append(0 if r.get("label") in _SAFE else 1)
            channels.append(r.get("channel"))
    return labels, channels
def _load_scores(logits_path):
    with open(logits_path) as fh:
        return [1.0 - json.loads(l)["p_safe"] for l in fh if l.strip()]
def eval_candidate(val_path, logits_path):
    labels, channels = _load_val_meta(val_path)
    scores = _load_scores(logits_path)
    assert len(labels) == len(scores), f"row mismatch: {len(labels)} vs {len(scores)}"
    dr = detection_rate_at_fpr(labels, scores, fpr_targets=(0.01,))["0.01"]
    thr = dr["threshold"]
    hit = defaultdict(int); tot = defaultdict(int)
    for lab, ch, sc in zip(labels, channels, scores):
        if lab == 1:
            tot[ch] += 1
            if sc > thr: hit[ch] += 1
    per_ch = {ch: (round(hit[ch] / tot[ch], 4) if tot[ch] else None) for ch in tot}
    return {"dr_at_1pct_fpr": round(dr["dr"], 4), "achieved_fpr": round(dr["achieved_fpr"], 4),
            "threshold": round(thr, 4), "per_channel_recall": per_ch}


## Preflight — augmented train + tokenizer boundary

In [ ]:
# ── Preflight: augmented-train stats + tokenizer boundary ───────────────
if MANIFEST.exists():
    man = json.loads(MANIFEST.read_text())
    bb, ba = man["benign_before"], man["benign_after"]
    print(f"benign len p90: {bb['len_p90']} -> {ba['len_p90']}   "
          f"added {man['added']['n_total']} long benign   "
          f"eval collisions removed: {man['dedup']['removed_eval_collisions']}")
else:
    print("(manifest not found — skipping augmentation summary)")

from transformers import AutoTokenizer
from models.prompt_template import load_model_config, format_training_example
_cfg = load_model_config(str(CONFIG_PATH))
_tok = AutoTokenizer.from_pretrained(_cfg.hf_id)
for _label in _cfg.labels:
    _p, _c = format_training_example(_cfg, "Summarize the following report.", _label)
    _pids = _tok(_p, add_special_tokens=False)["input_ids"]
    _cids = _tok(_c, add_special_tokens=False)["input_ids"]
    _lids = _tok(_label, add_special_tokens=False)["input_ids"]
    ok = (_pids + _cids)[:len(_pids + _lids)] == _pids + _lids and _cids[-1] == _tok.eos_token_id
    assert ok, f"tokenizer boundary broken for label={_label!r}"
    print(f"  boundary OK: label={_label:9s} n_prompt={len(_pids)} n_label={len(_lids)}")


## Fine-tune

In [ ]:
# ── Fine-tune (the long step; ~30-90 min on T4/L4) ────────────────
rc = run_train(CONFIG_PATH, OUTPUT_DIR, train_file=TRAIN_FILE, training_cfg=TRAINING_CFG)
assert rc == 0, f"training failed (rc={rc})"
if (OUTPUT_DIR / "train_summary.json").exists():
    print(json.dumps(json.loads((OUTPUT_DIR / "train_summary.json").read_text()), indent=2))


## Sanity — logit dumps

In [ ]:
# ── Sanity: logit dumps ────────────────────────────────────────
rep = sanity_logits(OUTPUT_DIR)
print(f"{BASE}: {rep}")
if not DRY_RUN:
    for stem in ("val", "cal"):
        assert rep[stem]["exists"] and rep[stem]["rows_ok"], f"{stem} row count wrong"
        assert rep[stem]["bad_rows"] == 0, f"{stem} out-of-range/NaN p_safe"
else:
    print("\n(DRY-RUN: asserts deferred to the full run.)")


## Gate preview — val benign FPR

In [ ]:
# ── Gate preview: val benign FPR + DR@1%FPR vs DataSentinel ~62-65% ─────────
if DRY_RUN:
    print("DRY-RUN: no val_logits to gate on. Set RUN_MODE='full' on GPU first.")
else:
    m = eval_candidate(VAL_FILE, OUTPUT_DIR / "val_logits.jsonl")
    print(f"DR@1%FPR (val)   : {m['dr_at_1pct_fpr']:.4f}")
    print(f"achieved FPR     : {m['achieved_fpr']:.4f}  (target 1%)")
    print(f"per-channel recall @1%FPR: {m['per_channel_recall']}")
    print("\nDataSentinel-7B benign FPR baseline: ~0.62-0.65 (bead t6y).")
    print("A working stage 2 holds ~1% FPR at high injection recall on direct/tool-output.")
